# Fresh Apple vs Rotten Apple — GET324 Mini Project
MobileNetV2 transfer learning, Kaggle CPU.

In [ ]:
!pip show tensorflow numpy matplotlib seaborn scikit-learn pandas --quiet
# !pip install tensorflow numpy matplotlib seaborn scikit-learn pandas --quiet


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

print(tf.__version__)


In [ ]:
results_dir = "/kaggle/working/results/"
os.makedirs(results_dir, exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
print(gpus if gpus else 'No GPU, running on CPU')


Dataset tree under /kaggle/input

In [ ]:
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    if depth <= 4:
        print(root)
    if depth >= 4:
        dirs[:] = []


Recursive search for train/val/test folders — edit SEARCH_ROOT only if STEP above shows a different top-level folder.

In [ ]:
SEARCH_ROOT = "/kaggle/input"

SPLIT_NAMES = {
    'train': ['train', 'training'],
    'val':   ['val', 'valid', 'validation'],
    'test':  ['test', 'testing'],
}

def find_split_dirs(search_root):
    search_root = Path(search_root)
    candidates = {'train': [], 'val': [], 'test': []}
    for dirpath, dirnames, _files in os.walk(search_root):
        base_name = Path(dirpath).name.lower()
        for key, names in SPLIT_NAMES.items():
            if base_name in names:
                candidates[key].append(Path(dirpath))
    result = {}
    for key, paths in candidates.items():
        result[key] = min(paths, key=lambda p: len(p.parts)) if paths else None
    return result

splits = find_split_dirs(SEARCH_ROOT)
train_dir = splits['train']
val_dir   = splits['val']
test_dir  = splits['test']

print('train_dir:', train_dir)
print('val_dir  :', val_dir)
print('test_dir :', test_dir)


In [ ]:
def count_images(d):
    if d is None or not Path(d).exists():
        return 0
    exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    return sum(len(list(Path(d).rglob(e))) for e in exts)

for split, d in [('train', train_dir), ('val', val_dir), ('test', test_dir)]:
    print(split, d, count_images(d))


In [ ]:
IMAGE_HEIGHT = 224
IMAGE_WIDTH  = 224
BATCH_SIZE   = 32
EPOCHS       = 20
LR           = 1e-4


Load datasets. If no separate val/test split exists, carve 15%/15% off train.

In [ ]:
if train_dir is None:
    raise FileNotFoundError("No 'train' folder found. Check SEARCH_ROOT.")

if val_dir is not None and test_dir is not None:
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        train_dir, image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
        batch_size=BATCH_SIZE, label_mode='binary', seed=SEED, shuffle=True,
    )
    val_dataset = tf.keras.utils.image_dataset_from_directory(
        val_dir, image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
        batch_size=BATCH_SIZE, label_mode='binary', seed=SEED, shuffle=False,
    )
    test_dataset = tf.keras.utils.image_dataset_from_directory(
        test_dir, image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
        batch_size=BATCH_SIZE, label_mode='binary', seed=SEED, shuffle=False,
    )
else:
    full_train = tf.keras.utils.image_dataset_from_directory(
        train_dir, image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
        batch_size=BATCH_SIZE, label_mode='binary', seed=SEED, shuffle=True,
    )
    n_batches = tf.data.experimental.cardinality(full_train).numpy()
    val_batches  = max(1, int(n_batches * 0.15))
    test_batches = max(1, int(n_batches * 0.15))

    test_dataset  = full_train.take(test_batches)
    remaining     = full_train.skip(test_batches)
    val_dataset   = remaining.take(val_batches)
    train_dataset = remaining.skip(val_batches)


In [ ]:
class_names = train_dataset.class_names if hasattr(train_dataset, 'class_names') else ['class_0', 'class_1']
print(class_names)


In [ ]:
for images, labels in train_dataset.take(1):
    fixed_images = images.numpy()
    fixed_labels = labels.numpy()

plt.figure(figsize=(12, 8))
for i in range(min(16, len(fixed_images))):
    ax = plt.subplot(4, 4, i + 1)
    plt.imshow(fixed_images[i].astype('uint8'))
    plt.title(class_names[int(fixed_labels[i][0])], fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.savefig(results_dir + 'sample_images.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset   = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset  = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)


In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15),
], name='data_augmentation')


In [ ]:
for images, _ in train_dataset.take(1):
    sample = images[0:1]
    break

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
axes[0].imshow(sample[0].numpy().astype('uint8'))
axes[0].set_title('Original')
axes[0].axis('off')
for i in range(1, 6):
    aug_img = data_augmentation(sample, training=True)[0].numpy().astype('uint8')
    axes[i].imshow(aug_img)
    axes[i].set_title(f'Aug {i}')
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(results_dir + 'augmentation_examples.png', dpi=120, bbox_inches='tight')
plt.show()


Callbacks and evaluation helpers. `evaluate_model` now returns loss/accuracy/precision/recall/f1/auc together so model choice isn't based on accuracy alone.

In [ ]:
def make_callbacks(name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=f'/kaggle/working/models/{name}_best.keras',
            monitor='val_loss', mode='min', save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=6, restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1,
        ),
    ]


def plot_learning_curves(history, title='Learning Curves'):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc, label='Train Accuracy')
    plt.plot(epochs_range, val_acc, label='Val Accuracy', linestyle='--')
    plt.legend(); plt.title(f'{title} — Accuracy'); plt.grid(alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Train Loss')
    plt.plot(epochs_range, val_loss, label='Val Loss', linestyle='--')
    plt.legend(); plt.title(f'{title} — Loss'); plt.grid(alpha=0.3)

    plt.tight_layout()
    safe_title = title.lower().replace(' ', '_')
    plt.savefig(results_dir + f'{safe_title}_curves.png', dpi=120, bbox_inches='tight')
    plt.show()


def evaluate_model(model, dataset, class_names, title='Model', plot=True):
    y_true, y_pred, y_prob = [], [], []
    for images, labels in dataset:
        probs = model.predict(images, verbose=0).ravel()
        y_prob.append(probs)
        y_pred.append((probs >= 0.5).astype(int))
        y_true.append(labels.numpy().ravel().astype(int))

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_prob = np.concatenate(y_prob)

    loss, acc, precision, recall, auc = model.evaluate(dataset, verbose=0)
    f1 = f1_score(y_true, y_pred)

    metrics = {
        'loss': loss, 'accuracy': acc, 'precision': precision,
        'recall': recall, 'f1': f1, 'auc': auc,
    }

    print(f'\n{title}')
    for k, v in metrics.items():
        print(f'{k:>10}: {v:.4f}')
    print()
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    if plot:
        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, cmap='Blues', cbar=False, annot=True, fmt='d',
                    xticklabels=class_names, yticklabels=class_names,
                    annot_kws={'fontsize': 11, 'fontweight': 'bold'})
        plt.title(f'{title} — Confusion Matrix')
        plt.xlabel('Predicted'); plt.ylabel('True')
        plt.tight_layout()
        safe_title = title.lower().replace(' ', '_')
        plt.savefig(results_dir + f'{safe_title}_confusion.png', dpi=120, bbox_inches='tight')
        plt.show()

    return metrics


Model. `label_smoothing=0.1` on the loss and L2 + dropout on the head stop the sigmoid from saturating to exact 0/1 — a model this size on a few thousand images shouldn't ever report 100% confidence.

In [ ]:
OFFLINE_WEIGHTS_PATH = None
# e.g. "/kaggle/input/tf-keras-pretrained-model-weights/mobilenet_v2_weights_tf_dim_ordering_tf_kernels_1.0_224_no_top.h5"

INPUT_SHAPE = (IMAGE_HEIGHT, IMAGE_WIDTH, 3)

def build_transfer_model(input_shape, augmentation, offline_weights_path=None):
    if offline_weights_path:
        base_model = tf.keras.applications.MobileNetV2(
            weights=None, input_shape=input_shape, include_top=False,
        )
        base_model.load_weights(offline_weights_path)
    else:
        try:
            base_model = tf.keras.applications.MobileNetV2(
                weights='imagenet', input_shape=input_shape, include_top=False,
            )
        except Exception as e:
            raise RuntimeError(
                "Could not download ImageNet weights (Kaggle internet likely OFF). "
                "Enable Internet in notebook Settings, or set OFFLINE_WEIGHTS_PATH."
            ) from e

    preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input
    base_model.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    x = augmentation(inputs)
    x = preprocess_fn(x)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation='relu',
                               kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    return tf.keras.Model(inputs, outputs, name='mobilenetv2_transfer'), base_model


tl_model, base_model = build_transfer_model(INPUT_SHAPE, data_augmentation, OFFLINE_WEIGHTS_PATH)
tl_model.summary()


In [ ]:
tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')],
)

tl_history = tl_model.fit(
    train_dataset, validation_data=val_dataset, epochs=EPOCHS,
    callbacks=make_callbacks('feature_extraction'), verbose=1,
)


In [ ]:
plot_learning_curves(tl_history, 'Feature Extraction')
fe_val_metrics = evaluate_model(tl_model, val_dataset, class_names, 'Feature Extraction (val)')


Fine-tuning — unfreeze the top 30 backbone layers, drop the learning rate 10x.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

FINETUNE_LR = LR / 10

tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=FINETUNE_LR),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')],
)

ft_history = tl_model.fit(
    train_dataset, validation_data=val_dataset, epochs=10,
    callbacks=make_callbacks('finetuned'), verbose=1,
)


In [ ]:
plot_learning_curves(ft_history, 'Fine-Tuned')
ft_val_metrics = evaluate_model(tl_model, val_dataset, class_names, 'Fine-Tuned (val)')


Model selection. Compare both checkpoints on the validation set across loss/accuracy/precision/recall/f1/auc instead of picking by accuracy alone, then confirm the choice on the untouched test set.

In [ ]:
fe_model = tf.keras.models.load_model('/kaggle/working/models/feature_extraction_best.keras')
ft_model = tf.keras.models.load_model('/kaggle/working/models/finetuned_best.keras')

fe_metrics = evaluate_model(fe_model, val_dataset, class_names, 'Feature Extraction checkpoint (val)', plot=False)
ft_metrics = evaluate_model(ft_model, val_dataset, class_names, 'Fine-Tuned checkpoint (val)', plot=False)

comparison = pd.DataFrame([fe_metrics, ft_metrics], index=['feature_extraction', 'fine_tuned'])
print(comparison)

if ft_metrics['loss'] <= fe_metrics['loss']:
    final_model = ft_model
    final_name = 'fine_tuned'
else:
    final_model = fe_model
    final_name = 'feature_extraction'

print(f'\nSelected: {final_name} (lower validation loss)')


In [ ]:
test_metrics = evaluate_model(final_model, test_dataset, class_names, f'{final_name} (test)')


In [ ]:
def predict_image(img_path, model, class_names, image_size=(IMAGE_HEIGHT, IMAGE_WIDTH)):
    img = tf.keras.utils.load_img(img_path, target_size=image_size)
    img_array = np.expand_dims(tf.keras.utils.img_to_array(img), axis=0)

    plt.figure(figsize=(3, 3))
    plt.imshow(img); plt.axis('off'); plt.show()

    prob = model.predict(img_array, verbose=0)[0][0]
    label = class_names[int(prob >= 0.5)]
    confidence = prob if prob >= 0.5 else 1 - prob
    print(f'{label} ({confidence:.4f})')

# predict_image('/kaggle/input/your-test-image.jpg', final_model, class_names)


In [ ]:
FINAL_MODEL_PATH = '/kaggle/working/model.keras'
final_model.save(FINAL_MODEL_PATH)
print(FINAL_MODEL_PATH, test_metrics)
